# Поступашки: EDA и аудит качества данных (base.xlsx)

**Задача 1 хакатона.** Ниже — не набор графиков, а последовательный аудит:
по каждому пункту задания сначала фиксируется **проблема/вопрос**, затем
**находка** (что показали данные) и **решение** (как я договорился это
трактовать дальше по кейсу).

Пять пунктов аудита:
1. Что считать покупкой, заказом и уникальным покупателем.
2. Как обрабатывать одновременную покупку нескольких курсов и пакеты.
3. Повторные покупки, продуктовые сочетания, динамика продаж и выручки.
4. Временные закономерности, всплески, провалы, аномалии.
5. Какие бизнес-вопросы решаемы текущими данными, а какие — принципиально нет.

Файл: `base.xlsx` — 795 строк, поля: `student_id` (обезличенный id),
`amount` (сумма строки), `course` (курс), `ts` (время оплаты).

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter
from scipy import stats

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

df = pd.read_excel('base.xlsx')
df.columns = ['student_id', 'amount', 'course', 'ts']
df['date'] = df['ts'].dt.date

print(df.shape)
print(df.dtypes)
df.head()

(795, 5)
student_id             int64
amount               float64
course                   str
ts            datetime64[us]
date                  object
dtype: object


   student_id  amount            course                  ts        date
0         437  8950.0            ML про 2026-08-04 09:08:31  2026-08-04
1         437  7890.0   Аналитика старт 2026-08-04 09:35:00  2026-08-04
2         443  8475.0  Линейная алгебра 2026-08-04 10:14:38  2026-08-04
3         443  8475.0        Мат анализ 2026-08-04 10:14:38  2026-08-04
4         430  8950.0            К ВУЗу 2026-08-04 13:38:06  2026-08-04

## 0. Технический аудит (предпосылка ко всему остальному)

Прежде чем говорить о бизнес-смысле, проверяем данные как таковые:
пропуски, полные дубли строк, диапазон дат, число уникальных студентов
и курсов.

In [ ]:
print("Пропуски по колонкам:")
print(df.isna().sum())
print()
print("Полных дублей строк:", df.duplicated().sum())
print("Уникальных student_id:", df.student_id.nunique())
print("Уникальных курсов:", df.course.nunique())
print("Диапазон дат:", df.ts.min(), "—", df.ts.max())

Пропуски по колонкам:
student_id    0
amount        0
course        0
ts            0
date          0
dtype: int64

Полных дублей строк: 0
Уникальных student_id: 606
Уникальных курсов: 18
Диапазон дат: 2026-08-04 09:08:31 — 2026-09-10 06:09:12


In [ ]:
df.course.value_counts()

course
Аналитика про          93
Аналитика старт        91
AI агенты              91
ML про                 90
Алгоритмы старт        88
ML старт               80
Backend про            55
Backend старт          39
Алгоритмы про          35
Алгоритмы              30
К ВУЗу                 26
Мат анализ             20
Линейная алгебра       19
Теория вероятностей    13
Data Science            9
АВ тестам               8
Data Engenering         4
Дискретка               4
Name: count, dtype: int64

## 1. Что считать покупкой, заказом и уникальным покупателем

**Проблема:** 795 строк файла — это не то же самое, что 795 "покупок".
Один платёж может состоять из нескольких строк (пакет курсов), а один
человек — совершить несколько разных платежей. Без явного определения
"заказа" все последующие метрики (выручка, AOV, конверсия) считаются
на глазок и разными людьми по-разному.

**Проверка:** группируем строки по `(student_id, ts)` — гипотеза в том,
что строки с одинаковым студентом и одинаковой секундой оплаты относятся
к одному платежу.

In [ ]:
orders = df.groupby(['student_id', 'ts']).agg(
    amount=('amount', 'sum'),
    n_items=('course', 'count'),
    date=('date', 'first'),
).reset_index()

orders

Строк в исходных данных: 795
Заказов (уникальных student_id + ts): 628

Распределение размера заказа (число курсов в одном платеже):
n_items
1    475
2    145
3      4
4      2
5      2
Name: count, dtype: int64

Заказов с >1 курсом (пакеты): 153
Строк, входящих в пакеты: 320 из 795


In [ ]:
print("Строк в исходных данных:", len(df))
print("Заказов (уникальных student_id + ts):", len(orders))

print("Распределение размера заказа (число курсов в одном платеже):")
print(orders.n_items.value_counts().sort_index())
print()
print("Заказов с >1 курсом (пакеты):", (orders.n_items > 1).sum())
print("Строк, входящих в пакеты:", df.groupby(['student_id', 'ts'])['course']
      .transform('size').gt(1).sum(), "из", len(df))

**Находка:** 795 строк → 628 заказов, 606 уникальных студентов. 153 из
628 заказов (24%) — пакеты из 2–5 курсов, это 320 строк (40% всех
строк).

**Решение:**
- **Покупка** = одна строка файла (один курс в составе платежа).
- **Заказ** = группа строк с одинаковыми `student_id` и `ts` (одна
  транзакция оплаты, независимо от числа курсов внутри).
- **Уникальный покупатель** = `student_id`.
- **Допущение**, которое я закладываю явно: секундной точности `ts`
  достаточно, чтобы не путать два разных заказа одного студента —
  при таком дневном объёме (максимум ~70 заказов/день на всю базу)
  коллизия по секундам у одного и того же студента маловероятна, но
  это предположение, а не доказанный факт, и его стоит проверить, если
  появится более точный лог.



## 2. Как обрабатывать одновременную покупку нескольких курсов и пакеты

**Проблема:** если считать `amount` ценой конкретного курса даже внутри
пакета, доход по каждому продукту будет посчитан неверно — нужно
понять, что вообще представляет собой сумма в пакетной строке.

**Проверка:** ищем дробные суммы (копейки) — если пакет продавался за
единую цену, а `amount` — это честная доля этой цены на курс, сумма по
строкам одного заказа должна сходиться в круглое число, а отдельные
строки — иметь копейки от деления.

In [ ]:
frac_mask = (df['amount'] % 1 != 0)
print(f"Строк с дробной суммой (не целое число рублей): {frac_mask.sum()} из {len(df)}")
example = df[(df.student_id == 58)].sort_values('ts')
print()
print("Пример — студент 58, один платёж за 3 курса:")
print(example[['student_id', 'course', 'amount', 'ts']])
print()
print("Сумма по строкам:", example['amount'].sum(), "— целое число, хотя каждая строка дробная.")

Строк с дробной суммой (не целое число рублей): 21 из 795

Пример — студент 58, один платёж за 3 курса:
     student_id           course   amount                  ts
117          58    Backend старт  5463.33 2026-08-09 16:12:46
118          58         ML старт  5463.33 2026-08-09 16:12:46
119          58  Алгоритмы старт  5463.34 2026-08-09 16:12:46

Сумма по строкам: 16390.0 — целое число, хотя каждая строка дробная.


**Находка:** 5463.33 + 5463.33 + 5463.34 = 16390.00 — сумма заказа
поделена поровну на число курсов, остаток округления ушёл в последнюю
строку. Значит **`amount` пакетной строки — это доля суммы заказа, а
не цена конкретного курса** внутри пакета.

**Решение:**

- Выручку и средний чек (AOV) считаем по заказу — то есть по сумме всех строк одной покупки (student_id, ts), а не по каждой строке отдельно. Иначе один платёж за несколько курсов посчитается как несколько разных заказов, и средний чек занизится.

- Сколько именно заработал каждый отдельный курс внутри пакета — не считаем. В файле это просто "сумма заказа / число курсов", а не настоящая цена курса, поэтому выдавать эту цифру за реальную выручку курса нельзя — это будет придумано, а не измерено.

- Для анализа "что покупают вместе" курсы внутри пакета всё равно учитываем — сам факт совместной покупки реален, просто это не про деньги по курсу, а про сочетание продуктов (см. п. 3).




## 3. Повторные покупки, продуктовые сочетания, динамика продаж и выручки

### 3.1 Повторные покупки

**Проблема:** нужно понять, насколько бизнес держится на повторных
клиентах — это влияет на то, стоит ли считать retention/LTV частью
будущей ROMI-модели.

In [ ]:
per_student_orders = df.groupby('student_id')['ts'].nunique()
repeat_ids = per_student_orders[per_student_orders > 1].index

print("Распределение числа заказов на студента:")
print(per_student_orders.value_counts().sort_index())

print(f"Повторных покупателей: {len(repeat_ids)} из {df.student_id.nunique()} "
      f"({len(repeat_ids) / df.student_id.nunique():.1%})")
rev_repeat = df[df.student_id.isin(repeat_ids)]['amount'].sum()
print(f"Их доля в общей сумме amount: {rev_repeat / df['amount'].sum():.1%}")

Распределение числа заказов на студента:
ts
1    586
2     19
4      1
Name: count, dtype: int64

Повторных покупателей: 20 из 606 (3.3%)
Их доля в общей сумме amount: 6.3%


**Находка:** 20 из 606 студентов (3.3%) сделали больше одного заказа,
на них приходится 6.3% суммарного `amount`.

**Решение:** повторная покупка редкое исключение. На нынешнем объёме данных **не закладываю retention/LTV как
основу ROMI-модели** — с 20 наблюдениями это статистически ненадёжно



### 3.2 Продуктовые сочетания

**Проблема:** нужно понять, случайны ли пакеты или это осознанная
продуктовая логика (апселл/кросс-селл) — это влияет на то, как в
будущем считать ROMI кампании, которая рекламирует один курс, а
покупают два.

In [ ]:
combo_counter = Counter()
for (sid, ts), g in df.groupby(['student_id', 'ts']):
    if len(g) > 1:
        for pair in combinations(sorted(g.course.tolist()), 2):
            combo_counter[pair] += 1

for pair, cnt in combo_counter.most_common(10):
    print(cnt, pair)

24 ('ML старт', 'Алгоритмы старт')
23 ('Аналитика про', 'Аналитика старт')
18 ('AI агенты', 'ML про')
13 ('ML про', 'ML старт')
10 ('Алгоритмы старт', 'Аналитика старт')
10 ('Алгоритмы про', 'Алгоритмы старт')
9 ('Backend старт', 'Алгоритмы старт')
6 ('AI агенты', 'Backend про')
5 ('Линейная алгебра', 'Мат анализ')
5 ('Backend про', 'Backend старт')


**Находка:** топ-связки — не случайны: "старт + про" одного трека
(апселл: Аналитика, Backend, Алгоритмы) и смежные треки (ML + Алгоритмы,
AI агенты + ML).

**Решение:** пакеты — это продуктовая механика (апселл/кросс-селл), а
не шум. При построении attribution/ROMI (Задачи 5–8) заказ нужно
считать по его полному составу курсов, а не только по "первому"
курсу: выручка одного рекламного касания часто равна не одному курсу, а сумме пакета.

### 3.3 Динамика продаж и выручки

In [ ]:
daily = orders.groupby('date').agg(
    orders=('student_id', 'count'),
    revenue=('amount', 'sum'),
    items=('n_items', 'sum'),
).reset_index()
daily['dow'] = pd.to_datetime(daily['date']).dt.day_name()
daily

          date  orders    revenue  items        dow
0   2026-08-04       7   84130.00      9    Tuesday
1   2026-08-05       6   67110.00      8  Wednesday
2   2026-08-06       7   62445.00      8   Thursday
3   2026-08-07       5   54995.00      6     Friday
4   2026-08-08      35  275015.00     49   Saturday
5   2026-08-09      68  524091.67     92     Sunday
6   2026-08-10      29  205590.00     36     Monday
7   2026-08-11      12   96330.00     17    Tuesday
8   2026-08-12      14  114930.00     17  Wednesday
9   2026-08-13       9   72810.00      9   Thursday
10  2026-08-14       9   76690.00     10     Friday
11  2026-08-15       5   48345.00      7   Saturday
12  2026-08-16       5   43085.00      6     Sunday
13  2026-08-17       3   33185.00      4     Monday
14  2026-08-18       6   61110.00      7    Tuesday
15  2026-08-19       5   56380.00      7  Wednesday
16  2026-08-20       6   52585.00      6   Thursday
17  2026-08-21       3   28145.00      3     Friday
18  2026-08-

In [ ]:
order_totals = orders['amount']
print("Общая выручка (по заказам, без задвоения пакетов):",
      round(order_totals.sum(), 2))
print("Средний чек (AOV):", round(order_totals.mean(), 1))
print("Медианный чек:", order_totals.median())

Общая выручка (по заказам, без задвоения пакетов): 5904671.67
Средний чек (AOV): 9402.3
Медианный чек: 8950.0


**Находка:** выручка растёт неравномерно, с выраженными недельными
волнами (детали и статистическая проверка — в п. 4). AOV ≈ 9 402 ₽,
медиана 8 950 ₽ — совпадает с "базовой" ценой одного курса, то есть
типичный заказ — это один курс по полной цене, а пакеты и скидки
двигают среднее, но не медиану.

**Решение:** как целевую метрику продаж для дальнейших задач (прогноз,
ROMI) беру **выручку по заказам**, а не количество заказов как таковых, иначе рост числа
пакетов будет выглядеть как рост продаж без реального роста выручки.



## 4. Временные закономерности, всплески, провалы, аномалии

### 4.1 Всплески по дням

In [ ]:
daily

          date  orders    revenue  items        dow
0   2026-08-04       7   84130.00      9    Tuesday
1   2026-08-05       6   67110.00      8  Wednesday
2   2026-08-06       7   62445.00      8   Thursday
3   2026-08-07       5   54995.00      6     Friday
4   2026-08-08      35  275015.00     49   Saturday
5   2026-08-09      68  524091.67     92     Sunday
6   2026-08-10      29  205590.00     36     Monday
7   2026-08-11      12   96330.00     17    Tuesday
8   2026-08-12      14  114930.00     17  Wednesday
9   2026-08-13       9   72810.00      9   Thursday
10  2026-08-14       9   76690.00     10     Friday
11  2026-08-15       5   48345.00      7   Saturday
12  2026-08-16       5   43085.00      6     Sunday
13  2026-08-17       3   33185.00      4     Monday
14  2026-08-18       6   61110.00      7    Tuesday
15  2026-08-19       5   56380.00      7  Wednesday
16  2026-08-20       6   52585.00      6   Thursday
17  2026-08-21       3   28145.00      3     Friday
18  2026-08-

Видно три чётких всплеска, каждый выпадает на выходные:

- 08–10 авг: 35 → 68 → 29 заказов (норма буднего дня — 5–15)
- 22–24 авг: 26 → 39 → 19 заказов
- 04–06 сен: 26 → 49 → 45 заказов

Интервал между пиками — около двух недель. Похоже на регулярный цикл
запусков/промо в канале, а не на органический шум. Последний день
(10 сентября) неполный — данные обрываются в 06:09, его нужно
исключать из сравнений динамики, иначе он выглядит как обвал.

Проверяем эффект выходных формально (t-test уэлча, т.к. дисперсии
в группах разные).

**H0: разницы между количеством заказов в выходные и в будни нет**

**H1: разница есть**

In [ ]:
daily_full = daily[daily['date'] != daily['date'].max()].copy()
daily_full['is_weekend'] = pd.to_datetime(daily_full['date']).dt.dayofweek >= 5

weekend = daily_full.loc[daily_full.is_weekend, 'orders']
weekday = daily_full.loc[~daily_full.is_weekend, 'orders']

t_stat, p_val = stats.ttest_ind(weekend, weekday, equal_var=False)
df_welch = (weekend.var() / len(weekend) + weekday.var() / len(weekday)) ** 2 / (
    (weekend.var() / len(weekend)) ** 2 / (len(weekend) - 1)
    + (weekday.var() / len(weekday)) ** 2 / (len(weekday) - 1)
)
t_crit = stats.t.ppf(0.975, df_welch)

print(f"Выходные: n={len(weekend)}, mean={weekend.mean():.1f}, sd={weekend.std():.1f}")
print(f"Будни:    n={len(weekday)}, mean={weekday.mean():.1f}, sd={weekday.std():.1f}")
print(f"Welch t-test: t={t_stat:.2f}, df={df_welch:.1f}, "
      f"t_crit(α=0.05, двусторонний)={t_crit:.2f}, p={p_val:.5f}")

Выходные: n=10, mean=30.1, sd=20.9
Будни:    n=27, mean=12.1, sd=7.3
Welch t-test: t=2.67, df=9.8, t_crit(α=0.05, двусторонний)=2.23, p=0.02381


|t| = 2.67 > t_crit = 2.23 → различие значимо на уровне 5%.

**Решение:** |t| = 2.67 > t_crit = 2.23 (p ≈ 0.024) — различие значимо
на уровне 5%, это не случайный шум. "выходные/будни" и
"двухнедельный цикл" - рабочая гипотеза источника всплесков (лончи
или промо-рассылки в канале, таргетированные на выходные), но прямого журнала кампаний в этих данных нет (проверить её — задача восстановления маркетинговой истории, Задача 2).

### 4.2 Аномалии в цене

In [ ]:
print("Общая статистика по amount:")
print(df['amount'].describe())
print()
print("15 самых низких сумм:")
print(df.nsmallest(15, 'amount')[['student_id', 'amount', 'course', 'ts']])
print()
print("10 самых высоких сумм:")
print(df.nlargest(10, 'amount')[['student_id', 'amount', 'course', 'ts']])

Общая статистика по amount:
count      795.000000
mean      7427.259962
std       1827.497664
min        500.000000
25%       6490.000000
50%       7475.000000
75%       8950.000000
max      19350.000000
Name: amount, dtype: float64

15 самых низких сумм:
     student_id  amount               course                  ts
23          477   500.0  Теория вероятностей 2026-08-06 14:52:49
134         462   500.0      Алгоритмы старт 2026-08-09 16:45:22
486         437  1000.0            AI агенты 2026-08-30 16:16:42
714         366  2237.5            AI агенты 2026-09-06 15:18:37
715         366  2237.5               ML про 2026-09-06 15:18:37
716         366  2237.5             ML старт 2026-09-06 15:18:37
717         366  2237.5      Алгоритмы старт 2026-09-06 15:18:37
719          55  2237.5            AI агенты 2026-09-06 15:35:13
720          23  2237.5             ML старт 2026-09-06 15:39:44
721          51  2237.5               ML про 2026-09-06 15:46:24
724         180  2237.5      

**Находка / решение по каждой аномалии** (фиксирую как открытые вопросы
к бизнесу, а не тихо чищу или додумываю):

- **Кластер 2237.5 ₽** — 6 позиций (ML старт, ML про, AI агенты,
  Алгоритмы старт), все с 6 сентября 15:18 до 16:23. Один и тот же
  нестандартный ценник на разные продукты в узком часовом окне — похоже
  на разовую флеш-акцию/промокод. Это пример того, как по всплеску
  цены можно частично восстановить маркетинговую активность даже без
  прямого журнала рекламы (см. Задачу 2).
- **500 ₽** (2 платежа) — на порядок ниже минимальной цены курса
  (~2200 ₽ на всё остальное). Тест/сотрудник/промокод/ошибка — не
  определить по имеющимся данным.
- **Верхние выбросы**: 19 350 ₽ (Алгоритмы, один курс), 18 900 ₽ × 2
  (Теория вероятностей), 15 695 ₽ (Data Science) — не совпадают ни с
  одной известной комбинацией пакета.
- Цена одного и того же курса в принципе очень нестабильна (пример:
  «Алгоритмы» — от 4225 ₽ до 19 350 ₽). Решение: не считать "цену
  курса" константой ни в одной последующей модели — только фактическую
  сумму заказа.

## 5. Какие бизнес-вопросы решаемы текущими данными, а какие — нет

**Решаемо этими данными:**
- объём и динамика продаж/выручки по дням, включая статистически
  подтверждённые всплески;
- состав пакетов и апселл/кросс-селл связки между курсами;
- доля и вклад повторных покупателей;
- средний чек и его разброс;
- обнаружение подозрительных ценовых кластеров как косвенных следов
  промо-активности (без подтверждения дат из внешнего источника).

**Принципиально нерешаемо этими данными (нужны Задачи 2–5):**
- какая покупка от какого рекламного канала/размещения/креатива
  пришла — в `base.xlsx` нет ключа к источнику трафика вообще;
- ROMI и любая атрибуция — физически не на чем считать без данных о
  рекламных касаниях, стоимости размещений и хотя бы приблизительного
  tracking-ключа, связывающего рекламу с `student_id`;
- инкрементальность (что реклама добавила сверх органики) — без
  контрольной группы или эксперимента attribution ≠ incrementality
  (Задача 7), а способа его провести на исторических данных нет.